# 🕵️ Dia 21 — O ciclo OSINT, parte 3: processamento em um grafo no Maltego

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 21* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-21.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O grafo filtrado do Dia 20 tem 17 nós. Rafa aponta para `cdn.serraazul-pix.example`: *"Resolve para onde? E esse IP é de quem?"* Você não sabe. Hoje é um salto de cada vez, do domínio até o AS, com a fonte anotada em cada caixinha.

Neste notebook você vai:

1. **Modelar entidades e transforms tipados** (como o Maltego faz).
2. **Jogar "qual transform cabe aqui?"**.
3. **Expandir o grafo do caso em três saltos**, com o limite de 24 simulado.
4. **Auditar** três entidades ao acaso e, com rede, resolver um domínio real de exemplo.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. Tipos importam

No Maltego, um *transform* só aparece para o tipo de entidade para o qual foi escrito. Vamos imitar isso: cada *transform* declara o **tipo de entrada** que aceita, e a função recusa o resto. É o que obriga você a ser preciso sobre **o que** cada fato é.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)                 # frozen: entidade imutável, pode virar chave de dicionário
class Entidade:
    tipo: str                           # "Domain", "DNS Name", "IP Address", "Netblock", "AS"...
    valor: str                          # o texto da entidade

# Saída sintética dos transforms para o caso (mesma tabela de caso-continuo/dados/dia-21/resolucao-isca.csv)
# (linha, transform, entrada, tipo_entrada, saída, tipo_saída)
RESOLUCAO = [
 [
  1,
  "To DNS Name - NS (name server)",
  "serraazul-pix.example",
  "Domain",
  "ns1.dnsbarato.example",
  "NS Record"
 ],
 [
  2,
  "To DNS Name [Find common DNS names]",
  "serraazul-pix.example",
  "Domain",
  "www.serraazul-pix.example",
  "DNS Name"
 ],
 [
  3,
  "To DNS Name [Find common DNS names]",
  "serraazul-pix.example",
  "Domain",
  "admin.serraazul-pix.example",
  "DNS Name"
 ],
 [
  4,
  "To DNS Name [Find common DNS names]",
  "serraazul-pix.example",
  "Domain",
  "cdn.serraazul-pix.example",
  "DNS Name"
 ],
 [
  5,
  "To IP Address [DNS]",
  "ns1.dnsbarato.example",
  "NS Record",
  "198.51.100.53",
  "IP Address"
 ],
 [
  6,
  "To IP Address [DNS]",
  "www.serraazul-pix.example",
  "DNS Name",
  "203.0.113.77",
  "IP Address"
 ],
 [
  7,
  "To IP Address [DNS]",
  "admin.serraazul-pix.example",
  "DNS Name",
  "203.0.113.77",
  "IP Address"
 ],
 [
  8,
  "To IP Address [DNS]",
  "cdn.serraazul-pix.example",
  "DNS Name",
  "192.0.2.80",
  "IP Address"
 ],
 [
  9,
  "To IP Address [DNS]",
  "serraazul-seguranca.example",
  "Domain",
  "203.0.113.77",
  "IP Address"
 ],
 [
  10,
  "To IP Address [DNS]",
  "coopserra-azul.example",
  "Domain",
  "198.51.100.140",
  "IP Address"
 ],
 [
  11,
  "To IP Address [DNS]",
  "serraazull.example",
  "Domain",
  "198.51.100.200",
  "IP Address"
 ],
 [
  12,
  "To Netblocks [Using routing info]",
  "203.0.113.77",
  "IP Address",
  "203.0.113.64/26",
  "Netblock"
 ],
 [
  13,
  "To Netblocks [Using routing info]",
  "192.0.2.80",
  "IP Address",
  "192.0.2.64/26",
  "Netblock"
 ],
 [
  14,
  "To Netblocks [Using routing info]",
  "198.51.100.53",
  "IP Address",
  "198.51.100.0/26",
  "Netblock"
 ],
 [
  15,
  "To Netblocks [Using routing info]",
  "198.51.100.140",
  "IP Address",
  "198.51.100.128/26",
  "Netblock"
 ],
 [
  16,
  "To Netblocks [Using routing info]",
  "198.51.100.200",
  "IP Address",
  "198.51.100.192/26",
  "Netblock"
 ],
 [
  17,
  "To AS Number",
  "203.0.113.64/26",
  "Netblock",
  "AS64501 HOSTBARATO-EXEMPLO",
  "AS"
 ],
 [
  18,
  "To AS Number",
  "192.0.2.64/26",
  "Netblock",
  "AS64500 CDN-EXEMPLO",
  "AS"
 ],
 [
  19,
  "To AS Number",
  "198.51.100.0/26",
  "Netblock",
  "AS64502 DNSBARATO-EXEMPLO",
  "AS"
 ],
 [
  20,
  "To AS Number",
  "198.51.100.128/26",
  "Netblock",
  "AS64503 NUVEM-EXEMPLO",
  "AS"
 ],
 [
  21,
  "To AS Number",
  "198.51.100.192/26",
  "Netblock",
  "AS64504 ESTACIONAMENTO-EXEMPLO",
  "AS"
 ]
]
REVERSE_IP_77 = ["serraazul-pix.example", "serraazul-seguranca.example", "cliente000-hostbarato.example", "cliente001-hostbarato.example", "cliente002-hostbarato.example", "cliente003-hostbarato.example", "cliente004-hostbarato.example", "cliente005-hostbarato.example", "cliente006-hostbarato.example", "cliente007-hostbarato.example", "cliente008-hostbarato.example", "cliente009-hostbarato.example", "cliente010-hostbarato.example", "cliente011-hostbarato.example", "cliente012-hostbarato.example", "cliente013-hostbarato.example", "cliente014-hostbarato.example", "cliente015-hostbarato.example", "cliente016-hostbarato.example", "cliente017-hostbarato.example", "cliente018-hostbarato.example", "cliente019-hostbarato.example", "cliente020-hostbarato.example", "cliente021-hostbarato.example"]

# Que tipos cada transform aceita (o "menu do botão direito")
ACEITA = {
    "To DNS Name - NS (name server)": {"Domain"},
    "To DNS Name [Find common DNS names]": {"Domain"},
    "To IP Address [DNS]": {"DNS Name", "NS Record", "Domain"},
    "To Netblocks [Using routing info]": {"IP Address"},
    "To AS Number": {"Netblock"},
    "To DNS Name [Reverse IP]": {"IP Address"},
}
LIMITE = 24                             # máximo de resultados por execução no plano Basic

def rodar_transform(nome, ent):
    """Devolve [(Entidade nova, nota de proveniência)], ou levanta erro se o tipo não serve."""
    if ent.tipo not in ACEITA[nome]:
        raise TypeError(f"'{nome}' não aparece no menu de uma entidade {ent.tipo}")
    if nome == "To DNS Name [Reverse IP]" and ent.valor == "203.0.113.77":
        saidas = [(Entidade("DNS Name", d), f"reverse IP, linha {22 + k}") for k, d in enumerate(REVERSE_IP_77)]
    else:
        saidas = [(Entidade(ts, s), f"{t}, linha {n}") for n, t, e, te, s, ts in RESOLUCAO
                  if t == nome and e == ent.valor]
    return saidas[:LIMITE]              # o plano corta em 24

print(rodar_transform("To IP Address [DNS]", Entidade("DNS Name", "www.serraazul-pix.example")))
try:
    rodar_transform("To AS Number", Entidade("IP Address", "203.0.113.77"))
except TypeError as erro:
    print("Recusado:", erro)

## 2. 🎮 Qual transform cabe nesta entidade?

Para cada entidade, escolha o número do *transform* que faz o **próximo salto** da cadeia do dia (domínio → nome DNS → IP → netblock → AS).

In [ ]:
OPCOES = ["To DNS Name [Find common DNS names]", "To IP Address [DNS]",
          "To Netblocks [Using routing info]", "To AS Number"]
DESAFIOS = [
    (Entidade("Domain", "serraazul-pix.example"), 0),
    (Entidade("DNS Name", "cdn.serraazul-pix.example"), 1),
    (Entidade("IP Address", "192.0.2.80"), 2),
    (Entidade("Netblock", "192.0.2.64/26"), 3),
]
MODO_JOGO = True
acertos = 0
for ent, certo in DESAFIOS:
    print(f"\n{ent.tipo}: {ent.valor}")
    for i, op in enumerate(OPCOES, start=1):
        print(f"   {i}. {op}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3", "4"):
            r = input("   Próximo salto (1-4): ").strip()
        acertos += (int(r) - 1 == certo)
        print("   ✅" if int(r) - 1 == certo else "   ❌", end="")
    print(f"   Resposta: {OPCOES[certo]}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(DESAFIOS)}")

## 3. Três saltos, com proveniência

A célula expande o grafo a partir da semente e dos três Similar Domain mantidos no Dia 20. **Toda aresta carrega a nota** (transform + linha da tabela), como o campo Notes do Maltego.

In [ ]:
from collections import defaultdict

SEMENTES = [Entidade("Domain", "serraazul-pix.example"),
            Entidade("Domain", "serraazul-seguranca.example"),   # Similar Domain mantido (Dia 20)
            Entidade("Domain", "coopserra-azul.example"),
            Entidade("Domain", "serraazull.example")]

arestas = []                                   # (de, para, nota)
fronteira = list(SEMENTES)                     # entidades a expandir no próximo salto
expandidas = set()                             # cada entidade é expandida UMA vez (economiza créditos)
for salto in range(1, 5):                      # domínio→DNS, DNS→IP, IP→bloco, bloco→AS
    nova = []
    for ent in fronteira:
        if ent in expandidas:
            continue                           # já rodamos transforms nela num salto anterior
        expandidas.add(ent)
        for nome in ACEITA:
            if ent.tipo in ACEITA[nome] and nome != "To DNS Name [Reverse IP]":
                for saida, nota in rodar_transform(nome, ent):
                    arestas.append((ent, saida, nota))
                    nova.append(saida)
    fronteira = list(dict.fromkeys(nova))      # remove repetidos, mantendo a ordem
    print(f"Salto {salto}: +{len(fronteira)} entidades")

# Desenha em texto, como o layout Hierarchical
filhos = defaultdict(list)
for a, b, nota in arestas:
    filhos[a].append((b, nota))
def desenhar(ent, nivel=0, vistos=()):
    for b, nota in filhos.get(ent, []):
        print("   " * nivel + f"└─ [{b.tipo}] {b.valor}   ({nota})")
        if b not in vistos:
            desenhar(b, nivel + 1, vistos + (b,))
for s in SEMENTES:
    print(f"\n[{s.tipo}] {s.valor}")
    desenhar(s, 1)

# Salva as arestas com a proveniência (é o artefato isca-saltos.csv)
import csv
with open("isca-saltos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["de_tipo", "de", "para_tipo", "para", "fonte"])
    for a, b, nota in arestas:
        w.writerow([a.tipo, a.valor, b.tipo, b.valor, nota])
print(f"\n💾 isca-saltos.csv: {len(arestas)} arestas, todas com fonte")

In [ ]:
# Quem compartilha o IP do site falso? (pergunta 1 do Rafa)
por_ip = defaultdict(set)
for a, b, nota in arestas:
    if b.tipo == "IP Address":
        por_ip[b.valor].add(a.valor)
for ip, nomes in por_ip.items():
    print(f"{ip:<16} ← {sorted(nomes)}")

# O reverse IP do 203.0.113.77 bate no limite?
rev = rodar_transform("To DNS Name [Reverse IP]", Entidade("IP Address", "203.0.113.77"))
qtd = len(rev)
nota = f"pelo menos {qtd}" if qtd == LIMITE else str(qtd)     # regra do original
print(f"\nReverse IP de 203.0.113.77: {nota} resultados (plano corta em {LIMITE})")
print("➡️ Compartilhar IP com 'pelo menos 24' vizinhos é ligação FRACA: serraazul-seguranca.example = provável, não confirmado.")

## 4. 🔍 Auditoria do Passo 9

Três entidades ao acaso. Para cada uma, o notebook mostra de onde ela veio. Se você estivesse no Maltego, seria a Detail View.

In [ ]:
import random
random.seed(21)                                   # semente fixa: todo mundo audita as mesmas 3 (troque para variar)
amostra = random.sample(arestas, 3)
for a, b, nota in amostra:
    print(f"[{b.tipo}] {b.valor}\n   veio de [{a.tipo}] {a.valor}\n   fonte: {nota}\n")

# Bônus com rede: o salto 2 de verdade, num domínio reservado de exemplo.
import socket
try:
    print("example.com resolve para:", sorted({i[4][0] for i in socket.getaddrinfo("example.com", 443)}))
except OSError as erro:
    print(f"(sem rede: {type(erro).__name__}; os domínios do caso são fictícios de qualquer forma)")

### 🧠 Camada 8

A seta entre `serraazul-seguranca.example` e `203.0.113.77` tem a mesma espessura da seta entre `www` e o mesmo IP. Uma vale muito menos que a outra. É a **ilusão de profundidade explicativa** (Rozenblit e Keil): o grafo bonito parece uma explicação. Se você não consegue descrever uma aresta numa frase com a fonte, ela não vai para o relatório.

## ✅ Antes de fechar o Dia 21

- [ ] Você sabe por que `To AS Number` não aparece numa entidade IP Address
- [ ] Grafo de três saltos com nota de fonte em toda aresta
- [ ] As três perguntas do Rafa respondidas nas notas
- [ ] "pelo menos 24" no reverse IP
- [ ] Três entidades auditadas

**Amanhã (Dia 22):** hubs, clusters e pontes. O `203.0.113.77` vai parecer o centro de tudo e não vai ser. E o `coopserra-azul.example`, que não divide nada de infraestrutura, vai se ligar ao golpe por outro caminho.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>